In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


# ============================================================
# 1. Floor Map Plotter
#    - calibration json 로드
#    - 도면 이미지 로드
#    - meter 좌표 -> pixel 좌표 변환
#    - 도면 위 trajectory overlay
# ============================================================
class FloorMapPlotter:
    def __init__(self, calibration_json_path: str):
        self.calibration_json_path = calibration_json_path

        with open(calibration_json_path, "r", encoding="utf-8") as f:
            self.info = json.load(f)

        self.image_path = self.info["image_path"]

        if not os.path.exists(self.image_path):
            raise FileNotFoundError(
                f"도면 이미지 파일을 찾을 수 없습니다: {self.image_path}\n"
                f"calibration json의 image_path를 확인하세요."
            )

        self.image = Image.open(self.image_path).convert("RGB")
        self.img_w, self.img_h = self.image.size

        self.m_per_px = float(self.info["m_per_px"])
        self.origin_px = np.array(self.info["origin_px"], dtype=float)
        self.theta_rad = float(self.info["theta_rad"])
        self.enu_y_axis = bool(self.info.get("enu_y_axis", True))

        # calibration 코드와 동일한 회전 행렬
        c = math.cos(-self.theta_rad)
        s = math.sin(-self.theta_rad)

        self.R = np.array([
            [c, -s],
            [s,  c],
        ], dtype=float)

        print("========== Floor Map Loaded ==========")
        print(f"image_path : {self.image_path}")
        print(f"image size : {self.img_w} x {self.img_h}")
        print(f"m_per_px   : {self.m_per_px}")
        print(f"px_per_m   : {1.0 / self.m_per_px}")
        print(f"origin_px  : {self.origin_px}")
        print(f"theta_deg  : {np.degrees(self.theta_rad):.4f}")
        print(f"enu_y_axis : {self.enu_y_axis}")
        print("======================================")

    def meter_to_px(self, path_m):
        """
        meter 좌표 경로를 image pixel 좌표로 변환

        Parameters
        ----------
        path_m : np.ndarray, shape (N, 2)
            [[x_m, y_m],
             [x_m, y_m],
             ...]

        Returns
        -------
        px : np.ndarray
        py : np.ndarray
        """
        path_m = np.asarray(path_m, dtype=float)

        if path_m.ndim != 2 or path_m.shape[1] != 2:
            raise ValueError("path_m은 shape (N, 2)의 배열이어야 합니다.")

        pts_px = []

        for X, Y in path_m:
            # ENU 좌표계 사용 시, meter 좌표의 +Y는 이미지 위쪽 방향
            if self.enu_y_axis:
                Y = -Y

            dp_rot = np.array([X, Y], dtype=float) / self.m_per_px
            dp = self.R.T @ dp_rot
            p = dp + self.origin_px

            pts_px.append(p)

        pts_px = np.asarray(pts_px, dtype=float)

        return pts_px[:, 0], pts_px[:, 1]

    def get_path_from_df(self, df, x_col, y_col, x_flag=False, y_flag=False):
        """
        DataFrame에서 x, y 컬럼을 읽어서 path_m 생성
        """
        if x_col not in df.columns:
            raise KeyError(f"'{x_col}' 컬럼이 DataFrame에 없습니다. 현재 컬럼: {list(df.columns)}")

        if y_col not in df.columns:
            raise KeyError(f"'{y_col}' 컬럼이 DataFrame에 없습니다. 현재 컬럼: {list(df.columns)}")

        x = df[x_col].values.astype(float).copy()
        y = df[y_col].values.astype(float).copy()

        if x_flag:
            x = -x

        if y_flag:
            y = -y

        return np.column_stack([x, y])

    def draw_trajectory_on_floor_map(
        self,
        true_data,
        pred_data_s22u,
        pred_data_s20,
        pred_data_conv=None,
        true_x_col="x_m",
        true_y_col="y_m",
        pred_x_col="pred_X",
        pred_y_col="pred_Y",
        conv_x_col="pred_X",
        conv_y_col="pred_Y",
        x_flag=False,
        y_flag=False,
        conv_x_flag=False,
        conv_y_flag=False,
        title=None,
        save_path=None,
        dpi=300,
        figsize=(10, 8),
        crop_mode="auto",
        crop_margin_px=600,
        fixed_xlim_px=None,
        fixed_ylim_px=None,
        show_axis=False,
        show_legend=True,
    ):
        """
        도면 위에 Reference / AI-PDR S22U / AI-PDR S20+ / Conventional PDR trajectory를 그림

        crop_mode:
            "auto"  : 궤적 주변만 자동 crop
            "fixed" : fixed_xlim_px, fixed_ylim_px 사용
            "full"  : 도면 전체 표시
        """

        # ----------------------------------------------------
        # 1. meter path 구성
        # ----------------------------------------------------
        true_path_m = self.get_path_from_df(
            true_data,
            true_x_col,
            true_y_col,
            x_flag=False,
            y_flag=False,
        )

        pred_path_s22u_m = self.get_path_from_df(
            pred_data_s22u,
            pred_x_col,
            pred_y_col,
            x_flag=x_flag,
            y_flag=y_flag,
        )

        pred_path_s20_m = self.get_path_from_df(
            pred_data_s20,
            pred_x_col,
            pred_y_col,
            x_flag=x_flag,
            y_flag=y_flag,
        )

        conv_path_m = None
        if pred_data_conv is not None:
            conv_path_m = self.get_path_from_df(
                pred_data_conv,
                conv_x_col,
                conv_y_col,
                x_flag=conv_x_flag,
                y_flag=conv_y_flag,
            )

        # ----------------------------------------------------
        # 2. meter -> pixel 변환
        # ----------------------------------------------------
        true_px, true_py = self.meter_to_px(true_path_m)
        s22u_px, s22u_py = self.meter_to_px(pred_path_s22u_m)
        s20_px, s20_py = self.meter_to_px(pred_path_s20_m)

        if conv_path_m is not None:
            conv_px, conv_py = self.meter_to_px(conv_path_m)
        else:
            conv_px, conv_py = None, None

        # ----------------------------------------------------
        # 3. 표시 범위 설정
        # ----------------------------------------------------
        if crop_mode == "auto":
            px_list = [true_px, s22u_px, s20_px]
            py_list = [true_py, s22u_py, s20_py]

            if conv_px is not None:
                px_list.append(conv_px)
                py_list.append(conv_py)

            all_px = np.concatenate(px_list)
            all_py = np.concatenate(py_list)

            x_min = max(0, int(np.min(all_px) - crop_margin_px))
            x_max = min(self.img_w, int(np.max(all_px) + crop_margin_px))
            y_min = max(0, int(np.min(all_py) - crop_margin_px))
            y_max = min(self.img_h, int(np.max(all_py) + crop_margin_px))

            xlim = (x_min, x_max)
            ylim = (y_max, y_min)

        elif crop_mode == "fixed":
            if fixed_xlim_px is None or fixed_ylim_px is None:
                raise ValueError(
                    "crop_mode='fixed'일 때 fixed_xlim_px, fixed_ylim_px를 지정해야 합니다."
                )

            xlim = fixed_xlim_px
            ylim = fixed_ylim_px

        elif crop_mode == "full":
            xlim = (0, self.img_w)
            ylim = (self.img_h, 0)

        else:
            raise ValueError("crop_mode은 'auto', 'fixed', 'full' 중 하나여야 합니다.")

        # ----------------------------------------------------
        # 4. Plot
        # ----------------------------------------------------
        fig, ax = plt.subplots(figsize=figsize)

        ax.imshow(self.image)

        # Reference path
        ax.plot(
            true_px,
            true_py,
            color="blue",
            linewidth=3.0,
            linestyle="-",
            label="Reference Trajectory",
            zorder=3,
        )

        # AI-PDR S22U predicted path
        ax.plot(
            s22u_px,
            s22u_py,
            color="orange",
            linewidth=2.6,
            linestyle="--",
            label="AI-PDR (S22U)",
            zorder=4,
        )

        # AI-PDR S20+ predicted path
        ax.plot(
            s20_px,
            s20_py,
            color="green",
            linewidth=2.6,
            linestyle="--",
            label="AI-PDR (S20+)",
            zorder=4,
        )

        # Conventional PDR path
        if conv_px is not None:
            ax.plot(
                conv_px,
                conv_py,
                color="purple",
                linewidth=2.4,
                linestyle=":",
                label="Conventional PDR",
                zorder=4,
            )

        # Start point
        ax.scatter(
            true_px[0],
            true_py[0],
            color="red",
            edgecolors="black",
            linewidths=0.8,
            s=150,
            marker="s",
            label="Start & End Point",
            zorder=6,
        )


        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

        # 이미지 좌표계에서도 비율 유지
        ax.set_aspect("equal", adjustable="box")

        if title is not None:
            ax.set_title(title, fontsize=13)

        if show_axis:
            ax.set_xlabel("Image X (px)")
            ax.set_ylabel("Image Y (px)")
        else:
            ax.axis("off")

        if show_legend:
            ax.legend(
                loc="upper right",
                fontsize=12,
                framealpha=0.95,
                markerscale=1.35,
                handlelength=2.6,
                borderpad=0.8,
                labelspacing=0.5,
            )

        plt.tight_layout()

        if save_path is not None:
            plt.savefig(
                save_path,
                dpi=dpi,
                bbox_inches="tight",
                pad_inches=0.03,
            )
            print(f"[저장] {save_path}")

        plt.show()

        result = {
            "true_px": true_px,
            "true_py": true_py,
            "s22u_px": s22u_px,
            "s22u_py": s22u_py,
            "s20_px": s20_px,
            "s20_py": s20_py,
            "xlim": xlim,
            "ylim": ylim,
        }

        if conv_px is not None:
            result["conv_px"] = conv_px
            result["conv_py"] = conv_py

        return result


# ============================================================
# 2. 데이터 경로 설정
# ============================================================

BASE_DIR = os.getcwd()

TRUE_DATA_PATHS = [
    os.path.join(BASE_DIR, "results/true/test1_true.csv"),
    os.path.join(BASE_DIR, "results/true/test2_true.csv"),
]

PRED_TEST1_PATHS = [
    # test1
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_looking_left.csv"),   # 0
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_looking_left.csv"),   # 1

    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_looking_right.csv"),  # 2
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_looking_right.csv"),  # 3

    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_swing_left.csv"),     # 4
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_swing_left.csv"),     # 5

    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_swing_right.csv"),    # 6
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_swing_right.csv"),    # 7

    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_calling_left.csv"),   # 8
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_calling_left.csv"),   # 9

    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_calling_right.csv"),  # 10
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_calling_right.csv"),  # 11
]

PRED_TEST2_PATHS = [
    # test2
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_looking.csv"),  # 0
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_looking.csv"),  # 1

    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_swing.csv"),    # 2
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_swing.csv"),    # 3

    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_calling.csv"),  # 4
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_calling.csv"),  # 5
]

# 실제 저장 위치가 result/conventional_pdr_paths라면 아래 경로를 사용
CONV_PDR_PATHS = [
    os.path.join(BASE_DIR, "results/conv/test1_looking_left.csv"),   # 0
    os.path.join(BASE_DIR, "results/conv/test1_looking_right.csv"),  # 1
    os.path.join(BASE_DIR, "results/conv/test1_swing_left.csv"),     # 2
    os.path.join(BASE_DIR, "results/conv/test1_swing_right.csv"),    # 3
    os.path.join(BASE_DIR, "results/conv/test1_calling_left.csv"),   # 4
    os.path.join(BASE_DIR, "results/conv/test1_calling_right.csv"),  # 5

    os.path.join(BASE_DIR, "results/conv/test2_looking.csv"),        # 6
    os.path.join(BASE_DIR, "results/conv/test2_swing.csv"),          # 7
    os.path.join(BASE_DIR, "results/conv/test2_calling.csv"),        # 8
]

# 결과 저장 폴더
SAVE_DIR = os.path.join(BASE_DIR, "result", "floor_map_with_conventional_pdr")
os.makedirs(SAVE_DIR, exist_ok=True)


# ============================================================
# 3. 데이터 로드
# ============================================================

test1_true_df = pd.read_csv(TRUE_DATA_PATHS[0])
test2_true_df = pd.read_csv(TRUE_DATA_PATHS[1])

# -------------------------
# Test 1 - S22U
# -------------------------
test1_s22u_looking_left_df = pd.read_csv(PRED_TEST1_PATHS[0])
test1_s22u_looking_right_df = pd.read_csv(PRED_TEST1_PATHS[2])
test1_s22u_swing_left_df = pd.read_csv(PRED_TEST1_PATHS[4])
test1_s22u_swing_right_df = pd.read_csv(PRED_TEST1_PATHS[6])
test1_s22u_calling_left_df = pd.read_csv(PRED_TEST1_PATHS[8])
test1_s22u_calling_right_df = pd.read_csv(PRED_TEST1_PATHS[10])

# -------------------------
# Test 1 - S20+
# -------------------------
test1_s20_looking_left_df = pd.read_csv(PRED_TEST1_PATHS[1])
test1_s20_looking_right_df = pd.read_csv(PRED_TEST1_PATHS[3])
test1_s20_swing_left_df = pd.read_csv(PRED_TEST1_PATHS[5])
test1_s20_swing_right_df = pd.read_csv(PRED_TEST1_PATHS[7])
test1_s20_calling_left_df = pd.read_csv(PRED_TEST1_PATHS[9])
test1_s20_calling_right_df = pd.read_csv(PRED_TEST1_PATHS[11])

# -------------------------
# Test 2 - S22U
# -------------------------
test2_s22u_looking_df = pd.read_csv(PRED_TEST2_PATHS[0])
test2_s22u_swing_df = pd.read_csv(PRED_TEST2_PATHS[2])
test2_s22u_calling_df = pd.read_csv(PRED_TEST2_PATHS[4])

# -------------------------
# Test 2 - S20+
# -------------------------
test2_s20_looking_df = pd.read_csv(PRED_TEST2_PATHS[1])
test2_s20_swing_df = pd.read_csv(PRED_TEST2_PATHS[3])
test2_s20_calling_df = pd.read_csv(PRED_TEST2_PATHS[5])

# -------------------------
# Conventional PDR
# -------------------------
test1_conv_looking_left_df = pd.read_csv(CONV_PDR_PATHS[0])
test1_conv_looking_right_df = pd.read_csv(CONV_PDR_PATHS[1])
test1_conv_swing_left_df = pd.read_csv(CONV_PDR_PATHS[2])
test1_conv_swing_right_df = pd.read_csv(CONV_PDR_PATHS[3])
test1_conv_calling_left_df = pd.read_csv(CONV_PDR_PATHS[4])
test1_conv_calling_right_df = pd.read_csv(CONV_PDR_PATHS[5])

test2_conv_looking_df = pd.read_csv(CONV_PDR_PATHS[6])
test2_conv_swing_df = pd.read_csv(CONV_PDR_PATHS[7])
test2_conv_calling_df = pd.read_csv(CONV_PDR_PATHS[8])


# ============================================================
# 4. Plotter 생성
# ============================================================

floor_plotter = FloorMapPlotter(
    calibration_json_path="대학본관_3층_calibration.json"
)


# ============================================================
# 5. Test 1 전체 케이스 도면 위 저장
# ============================================================

test1_cases = [
    (
        "test1_looking_left",
        test1_s22u_looking_left_df,
        test1_s20_looking_left_df,
        test1_conv_looking_left_df,
        "Looking - Left Turn",
    ),
    (
        "test1_looking_right",
        test1_s22u_looking_right_df,
        test1_s20_looking_right_df,
        test1_conv_looking_right_df,
        "Looking - Right Turn",
    ),
    (
        "test1_swing_left",
        test1_s22u_swing_left_df,
        test1_s20_swing_left_df,
        test1_conv_swing_left_df,
        "Swing - Left Turn",
    ),
    (
        "test1_swing_right",
        test1_s22u_swing_right_df,
        test1_s20_swing_right_df,
        test1_conv_swing_right_df,
        "Swing - Right Turn",
    ),
    (
        "test1_calling_left",
        test1_s22u_calling_left_df,
        test1_s20_calling_left_df,
        test1_conv_calling_left_df,
        "Calling - Left Turn",
    ),
    (
        "test1_calling_right",
        test1_s22u_calling_right_df,
        test1_s20_calling_right_df,
        test1_conv_calling_right_df,
        "Calling - Right Turn",
    ),
]

for name, s22u_df, s20_df, conv_df, title in test1_cases:
    floor_plotter.draw_trajectory_on_floor_map(
        true_data=test1_true_df,
        pred_data_s22u=s22u_df,
        pred_data_s20=s20_df,
        pred_data_conv=conv_df,
        true_x_col="x_m",
        true_y_col="y_m",
        pred_x_col="x",
        pred_y_col="y",
        conv_x_col="x",
        conv_y_col="y",
        x_flag=True,
        y_flag=True,
        conv_x_flag=False,
        conv_y_flag=False,
        title=None,
        save_path=os.path.join(SAVE_DIR, f"{name}_floor_map.png"),
        dpi=300,
        figsize=(10, 8),
        crop_mode="full",
        show_axis=False,
        show_legend=True,
    )


# ============================================================
# 6. Test 2 전체 케이스 도면 위 저장
# ============================================================

test2_cases = [
    (
        "test2_looking",
        test2_s22u_looking_df,
        test2_s20_looking_df,
        test2_conv_looking_df,
        "Looking",
    ),
    (
        "test2_swing",
        test2_s22u_swing_df,
        test2_s20_swing_df,
        test2_conv_swing_df,
        "Swing",
    ),
    (
        "test2_calling",
        test2_s22u_calling_df,
        test2_s20_calling_df,
        test2_conv_calling_df,
        "Calling",
    ),
]

for name, s22u_df, s20_df, conv_df, title in test2_cases:
    floor_plotter.draw_trajectory_on_floor_map(
        true_data=test2_true_df,
        pred_data_s22u=s22u_df,
        pred_data_s20=s20_df,
        pred_data_conv=conv_df,
        true_x_col="x_m",
        true_y_col="y_m",
        pred_x_col="x",
        pred_y_col="y",
        conv_x_col="x",
        conv_y_col="y",
        x_flag=True,
        y_flag=True,
        conv_x_flag=True,
        conv_y_flag=True,
        title=None,
        save_path=os.path.join(SAVE_DIR, f"{name}_floor_map.png"),
        dpi=300,
        figsize=(10, 8),
        crop_mode="full",
        show_axis=False,
        show_legend=True,
    )

========== Floor Map Loaded ==========
image_path : 대학본관_3층.png
image size : 10240 x 5760
m_per_px   : 0.007874003693986267
px_per_m   : 127.00019441999314
origin_px  : [5191. 2340.]
theta_deg  : 0.0000
enu_y_axis : True


KeyError: "'pred_X' 컬럼이 DataFrame에 없습니다. 현재 컬럼: ['x', 'y', 'heading']"